# Modeling — LSTM (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-19-lstm-modeling-design.md`.
Desember 2025 tidak dibuka di notebook ini.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch

from utils import model_lstm as lstm
from utils import modeling_prep, walk_forward

df = pd.read_parquet(modeling_prep.MODEL_INPUT_FILE)
print(df.shape, torch.__version__, torch.backends.mps.is_available())

## Benchmark

Satu putaran dua-fit di fold 5 dengan `DEFAULT_PARAMS`, di CPU dan MPS.
Yang diukur: detik per epoch, epoch tempat early stopping mendarat, dan
peak RSS. Ketiganya yang mengisi rumus anggaran di §2.2 spec.

In [ ]:
import resource, time

frame = walk_forward.eligible_rows(df)
split = walk_forward.prepare_fold(frame, 5, prepared=True)
print('train', len(split['train']), 'valid', len(split['valid']))

benchmark = {}
for device_name in ('cpu', 'mps'):
    try:
        make = lstm.bind_panel(df, device_name=device_name)
    except ValueError as failure:
        print(device_name, 'dilewati:', failure)
        continue
    fit_predict = make(lstm.DEFAULT_PARAMS, quantile=lstm.QUANTILE)
    started = time.time()
    prediction = fit_predict(split['train'], split['valid'])
    elapsed = time.time() - started
    best_epoch = fit_predict.best_epochs[0]
    # elapsed covers both fits: best_epoch + patience epochs in the
    # first, best_epoch in the second.
    epochs_run = 2 * best_epoch + lstm.EARLY_STOPPING_EPOCHS
    benchmark[device_name] = {
        'wall_seconds': elapsed,
        'best_epoch': best_epoch,
        'sec_per_epoch': elapsed / epochs_run,
        'peak_rss_gb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9,
        'pred_mean': float(prediction.mean()),
        'pred_max': float(prediction.max()),
    }
    print(device_name, benchmark[device_name])

pd.DataFrame(benchmark).T

## Anggaran pencarian

`N` datang dari angka benchmark, bukan dari tebakan. Kalau rumusnya
jatuh di bawah 6, `candidate_budget` melempar ValueError — itu sinyal
untuk memperkecil ruang search, bukan menaikkan plafon 8 jam.

In [ ]:
DEVICE = min(benchmark, key=lambda name: benchmark[name]['sec_per_epoch'])
measured = benchmark[DEVICE]
N_CANDIDATES = lstm.candidate_budget(
    sec_per_epoch=measured['sec_per_epoch'],
    best_epoch=measured['best_epoch'],
)
print('device terpilih:', DEVICE)
print('sec_per_epoch  :', round(measured['sec_per_epoch'], 1))
print('best_epoch     :', measured['best_epoch'])
print('N_CANDIDATES   :', N_CANDIDATES)

## Pencarian hyperparameter

Fold 3 dan 5 saja, seed 42, kriteria pinball@0.9 gabungan berbobot
jumlah baris. Checkpoint di-flush tiap kandidat selesai.

In [ ]:
candidates = lstm.sample_search_space(N_CANDIDATES, seed=42)
search_results = lstm.run_search(
    df, candidates,
    checkpoint_path=lstm.SEARCH_FILE,
    device_name=DEVICE,
)
search_results.sort_values('pinball').head(10)

## Walk-forward final

Pemenang dijalankan ulang di kelima fold, lalu difit final.

In [ ]:
best = lstm.select_best(search_results, candidates)
lstm.save_best_params(best)
print(best)

make = lstm.bind_panel(df, device_name=DEVICE)
fit_predict = make(best, quantile=lstm.QUANTILE)
results = walk_forward.run_walk_forward(
    df, fit_predict, model_name='lstm', alpha=lstm.QUANTILE)
results.to_csv(lstm.RESULTS_FILE, index=False)
print('best_epoch per fold:', fit_predict.best_epochs)

In [ ]:
bundle = lstm.fit_final(df, best, device_name=DEVICE)
lstm.save_bundle(bundle)
print(bundle['best_epoch'], bundle['n_train'])

## Hasil

In [ ]:
results = pd.read_csv(lstm.RESULTS_FILE)
overall = results[results['group_col'].isna()]
for name in overall['model'].unique():
    print(name, round(walk_forward.pooled_metric(results, name), 4))

overall[overall['model'] == 'lstm']

## Head-to-head tiga arah

Sah karena ketiganya dinilai di baris identik — dijamin
`walk_forward.eligible_rows()`. Potongan kedua (fold 1, 2, 4) adalah
angka bersihnya: tidak ada model yang memakai fold itu untuk seleksi.

In [ ]:
from utils import model_random_forest as rf
from utils import model_xgboost as xgb

tables = {
    'lstm': pd.read_csv(lstm.RESULTS_FILE),
    'xgboost': pd.read_csv(xgb.RESULTS_FILE),
    'random_forest': pd.read_csv(rf.RESULTS_FILE),
}
rows = []
for name, table in tables.items():
    rows.append({
        'model': name,
        'pinball_5_folds': walk_forward.pooled_metric(table, name),
        'pinball_folds_124': walk_forward.pooled_metric(
            table, name, folds=(1, 2, 4)),
        'mae_5_folds': walk_forward.pooled_metric(table, name, metric='mae'),
        'coverage': walk_forward.pooled_metric(
            table, name, metric='coverage'),
    })
pd.DataFrame(rows).sort_values('pinball_folds_124')